
# Medición de rendimiento con `time.perf_counter()` en AppTickets

`time.perf_counter()` se utiliza para medir tiempos de ejecución en una aplicación orientada a objetos basada en tickets.

La intención no es “optimizar por optimizar”, sino aprender a responder preguntas como:

- ¿Cuánto tarda crear varios tickets?
- ¿Cuánto tarda validar muchos folios?
- ¿Cuánto tarda atender tickets?
- ¿Qué pasa cuando un método también escribe en un archivo de log?
- ¿Cómo comparar dos bloques de código de forma clara?

> `time.perf_counter()` es ideal para medir intervalos cortos de tiempo porque usa un contador de alta resolución.  
> Su valor absoluto no importa; lo importante es la diferencia entre el tiempo inicial y el tiempo final.



## 1. Importar módulos necesarios

Usaremos:

- `time`: para medir duración con `perf_counter()`.
- `os`: para revisar o eliminar el archivo de log.
- `datetime`, `re`, `ABC`, `abstractmethod` y `wraps`: porque forman parte de la aplicación de tickets.


In [1]:

from abc import ABC, abstractmethod
from functools import wraps
from datetime import datetime
import re
import time
import os
import pandas as pd



## 2. Aplicación base: AppTickets

La siguiente celda concentra la aplicación de tickets para que el cuaderno pueda ejecutarse de forma independiente.

La aplicación incluye:

- Clase abstracta `TicketBase`.
- Clase principal `Ticket`.
- Clases especializadas:
  - `TicketSoporte`
  - `TicketDesarrollo`
  - `TicketInfraestructura`
- Encapsulación con propiedades.
- Validación de folios con `@staticmethod`.
- Contador global con `@classmethod`.
- Decorador `registrar_accion` para guardar historial y escribir en archivo de log.

> Nota didáctica: como algunos métodos escriben en `registro_acciones.log`, las mediciones de esos métodos incluyen el costo de escritura a archivo.


In [2]:

def registrar_accion(funcion):
    @wraps(funcion)
    def envoltura(*args, **kwargs):
        objeto = args[0]

        fecha_hora = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        clase = objeto.__class__.__name__
        metodo = funcion.__name__
        folio = getattr(objeto, "folio", "N/A")
        estado_anterior = getattr(objeto, "estado", "N/A")

        resultado = funcion(*args, **kwargs)

        estado_nuevo = getattr(objeto, "estado", "N/A")

        registro = {
            "fecha_hora": fecha_hora,
            "clase": clase,
            "folio": folio,
            "metodo": metodo,
            "estado_anterior": estado_anterior,
            "estado_nuevo": estado_nuevo
        }

        if hasattr(objeto, "_historial"):
            objeto._historial.append(registro)

        with open("registro_acciones.log", "a", encoding="utf-8") as archivo:
            archivo.write(
                f"{fecha_hora} | "
                f"{clase} | "
                f"{folio} | "
                f"{metodo} | "
                f"{estado_anterior} -> {estado_nuevo}\n"
            )

        return resultado

    return envoltura


class TicketBase(ABC):

    @abstractmethod
    def atender(self):
        pass


class Ticket(TicketBase):

    ESTADOS_VALIDOS = ["Abierto", "En proceso", "Cerrado"]
    _contador_global = 0

    def __init__(self, folio, titulo, descripcion):
        self._historial = []

        self.folio = folio
        self.titulo = titulo
        self.descripcion = descripcion
        self.estado = "Abierto"

        Ticket._contador_global += 1

        self._registrar_evento_manual(
            "crear",
            "N/A",
            self.estado
        )

    @property
    def folio(self):
        return self._folio

    @folio.setter
    def folio(self, nuevo_folio):
        if not self.es_folio_valido(nuevo_folio):
            raise ValueError(f"Folio no válido: {nuevo_folio}")

        self._folio = nuevo_folio

    @property
    def titulo(self):
        return self._titulo

    @titulo.setter
    def titulo(self, nuevo_titulo):
        self._titulo = nuevo_titulo

    @property
    def descripcion(self):
        return self._descripcion

    @descripcion.setter
    def descripcion(self, nueva_descripcion):
        self._descripcion = nueva_descripcion

    @property
    def estado(self):
        return self._estado

    @estado.setter
    def estado(self, nuevo_estado):
        if nuevo_estado not in self.ESTADOS_VALIDOS:
            raise ValueError(f"Estado no válido: {nuevo_estado}")

        self._estado = nuevo_estado

    @property
    def historial(self):
        return tuple(self._historial)

    def _registrar_evento_manual(self, metodo, estado_anterior, estado_nuevo):
        registro = {
            "fecha_hora": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "clase": self.__class__.__name__,
            "folio": self.folio,
            "metodo": metodo,
            "estado_anterior": estado_anterior,
            "estado_nuevo": estado_nuevo
        }

        self._historial.append(registro)

    @registrar_accion
    def cambiar_estado(self, nuevo_estado):
        self.estado = nuevo_estado

    @registrar_accion
    def cerrar(self):
        self.estado = "Cerrado"

    @classmethod
    def total_tickets_creados(cls):
        return cls._contador_global

    @staticmethod
    def es_folio_valido(folio):
        return bool(re.fullmatch(r"^TK-?\d{3}$", folio))

    def registrar_atencion(self):
        self.estado = "En proceso"
        return f"Ticket {self.folio} marcado como En proceso."

    @registrar_accion
    def atender(self):
        return self.registrar_atencion()

    def __str__(self):
        return (
            f"{self.__class__.__name__} | "
            f"Folio: {self.folio} | "
            f"Título: {self.titulo} | "
            f"Estado: {self.estado}"
        )


class TicketSoporte(Ticket):

    @registrar_accion
    def atender(self):
        mensaje_base = super().atender()
        return (
            f"{mensaje_base} "
            f"El ticket de soporte {self.folio} está siendo atendido por mesa de ayuda."
        )


class TicketDesarrollo(Ticket):

    @registrar_accion
    def atender(self):
        mensaje_base = super().atender()
        return (
            f"{mensaje_base} "
            f"El ticket de desarrollo {self.folio} fue asignado al equipo de programación."
        )


class TicketInfraestructura(Ticket):

    @registrar_accion
    def atender(self):
        mensaje_base = super().atender()
        return (
            f"{mensaje_base} "
            f"El ticket de infraestructura {self.folio} fue enviado al área de servidores/redes."
        )



## 3. Primera medición simple con `perf_counter()`

La estructura básica es:

```python
inicio = time.perf_counter()
# código que queremos medir
fin = time.perf_counter()

duracion = fin - inicio
```

La diferencia `fin - inicio` representa el tiempo transcurrido en segundos.


In [3]:

inicio = time.perf_counter()

ticket = TicketSoporte(
    "TK-001",
    "Error en inicio de sesión",
    "El usuario no puede entrar al sistema."
)

fin = time.perf_counter()

duracion = fin - inicio

print(ticket)
print(f"Tiempo de creación: {duracion:.8f} segundos")


TicketSoporte | Folio: TK-001 | Título: Error en inicio de sesión | Estado: Abierto
Tiempo de creación: 0.00046880 segundos



## 4. Función auxiliar para medir bloques de código

Para no repetir la misma estructura muchas veces, crearemos una función llamada `medir_tiempo`.

Esta función recibe:

- una descripción;
- una función a ejecutar;
- argumentos opcionales para esa función.

Devuelve un diccionario con el resultado de la medición.


In [4]:

def medir_tiempo(descripcion, funcion, *args, **kwargs):
    inicio = time.perf_counter()
    resultado = funcion(*args, **kwargs)
    fin = time.perf_counter()

    return {
        "descripcion": descripcion,
        "segundos": fin - inicio,
        "resultado": resultado
    }



## 5. Medir validación de folios

La validación de folios es una operación pequeña, pero es buena para demostrar una idea importante:

> Cuando una operación es muy rápida, conviene ejecutarla muchas veces para obtener una medición más visible.

En este caso validaremos 900 folios correctos y 900 folios incorrectos.


In [5]:

def validar_folios_correctos(cantidad=900):
    contador_validos = 0

    for numero in range(1, cantidad + 1):
        folio = f"TK-{numero:03d}"

        if Ticket.es_folio_valido(folio):
            contador_validos += 1

    return contador_validos


def validar_folios_incorrectos(cantidad=900):
    contador_invalidos = 0

    for numero in range(1, cantidad + 1):
        folio = f"ABC-{numero:03d}"

        if not Ticket.es_folio_valido(folio):
            contador_invalidos += 1

    return contador_invalidos


medicion_validos = medir_tiempo(
    "Validar 900 folios correctos",
    validar_folios_correctos
)

medicion_invalidos = medir_tiempo(
    "Validar 900 folios incorrectos",
    validar_folios_incorrectos
)

print(medicion_validos)
print(medicion_invalidos)


{'descripcion': 'Validar 900 folios correctos', 'segundos': 0.0018419999978505075, 'resultado': 900}
{'descripcion': 'Validar 900 folios incorrectos', 'segundos': 0.0014054999919608235, 'resultado': 900}



## 6. Crear varios tickets y medir el tiempo

Ahora mediremos la creación de varios objetos.

Como el patrón de folio solo permite tres dígitos, usaremos como máximo 900 tickets para mantener folios válidos como `TK-001`, `TK-002`, ..., `TK-900`.


In [6]:

def crear_tickets(cantidad=300):
    tickets = []

    for numero in range(1, cantidad + 1):
        folio = f"TK-{numero:03d}"

        if numero % 3 == 1:
            ticket = TicketSoporte(
                folio,
                "Solicitud de soporte",
                "El usuario requiere apoyo de mesa de ayuda."
            )
        elif numero % 3 == 2:
            ticket = TicketDesarrollo(
                folio,
                "Solicitud de desarrollo",
                "Se requiere ajustar una funcionalidad del sistema."
            )
        else:
            ticket = TicketInfraestructura(
                folio,
                "Solicitud de infraestructura",
                "Se requiere revisar servidor o red."
            )

        tickets.append(ticket)

    return tickets


medicion_creacion = medir_tiempo(
    "Crear 300 tickets",
    crear_tickets,
    300
)

tickets = medicion_creacion["resultado"]

print(f"Tickets creados: {len(tickets)}")
print(f"Tiempo de creación: {medicion_creacion['segundos']:.8f} segundos")
print(tickets[0])
print(tickets[-1])


Tickets creados: 300
Tiempo de creación: 0.00445580 segundos
TicketSoporte | Folio: TK-001 | Título: Solicitud de soporte | Estado: Abierto
TicketInfraestructura | Folio: TK-300 | Título: Solicitud de infraestructura | Estado: Abierto



## 7. Medir atención de tickets

En esta aplicación, atender un ticket no solo cambia su estado. También pasa por el decorador `registrar_accion`.

Eso significa que al ejecutar `ticket.atender()` ocurren varias cosas:

1. Se consulta el estado anterior.
2. Se ejecuta la atención.
3. Se cambia el estado a `En proceso`.
4. Se agrega un registro al historial interno.
5. Se escribe una línea en `registro_acciones.log`.

Por eso esta medición no representa únicamente “cambiar un atributo”, sino una operación de negocio más completa.


In [7]:

def limpiar_archivo_log():
    if os.path.exists("registro_acciones.log"):
        os.remove("registro_acciones.log")


def atender_tickets(tickets):
    respuestas = []

    for ticket in tickets:
        respuestas.append(ticket.atender())

    return respuestas


limpiar_archivo_log()

medicion_atencion = medir_tiempo(
    "Atender 300 tickets",
    atender_tickets,
    tickets
)

print(f"Tickets atendidos: {len(medicion_atencion['resultado'])}")
print(f"Tiempo de atención: {medicion_atencion['segundos']:.8f} segundos")

if os.path.exists("registro_acciones.log"):
    print(f"Tamaño del archivo log: {os.path.getsize('registro_acciones.log')} bytes")


Tickets atendidos: 300
Tiempo de atención: 0.19341870 segundos
Tamaño del archivo log: 50200 bytes



## 8. Medir cierre de tickets

Cerraremos solo los primeros 100 tickets para observar otra operación decorada.

La operación `cerrar()` también registra historial y escribe en archivo de log.


In [8]:

def cerrar_tickets(tickets):
    for ticket in tickets:
        ticket.cerrar()

    return len(tickets)


primeros_100 = tickets[:100]

medicion_cierre = medir_tiempo(
    "Cerrar 100 tickets",
    cerrar_tickets,
    primeros_100
)

print(f"Tickets cerrados: {medicion_cierre['resultado']}")
print(f"Tiempo de cierre: {medicion_cierre['segundos']:.8f} segundos")

print(primeros_100[0])
print(primeros_100[-1])


Tickets cerrados: 100
Tiempo de cierre: 0.04131770 segundos
TicketSoporte | Folio: TK-001 | Título: Solicitud de soporte | Estado: Cerrado
TicketSoporte | Folio: TK-100 | Título: Solicitud de soporte | Estado: Cerrado



## 9. Comparar mediciones en una tabla

Una buena práctica es no quedarse solo con impresiones individuales.

Cuando se comparan varias mediciones, una tabla permite ver con mayor claridad qué operación tomó más tiempo.


In [9]:

resultados = [
    {
        "Operación": medicion_validos["descripcion"],
        "Tiempo (segundos)": medicion_validos["segundos"]
    },
    {
        "Operación": medicion_invalidos["descripcion"],
        "Tiempo (segundos)": medicion_invalidos["segundos"]
    },
    {
        "Operación": medicion_creacion["descripcion"],
        "Tiempo (segundos)": medicion_creacion["segundos"]
    },
    {
        "Operación": medicion_atencion["descripcion"],
        "Tiempo (segundos)": medicion_atencion["segundos"]
    },
    {
        "Operación": medicion_cierre["descripcion"],
        "Tiempo (segundos)": medicion_cierre["segundos"]
    }
]

df_resultados = pd.DataFrame(resultados)
df_resultados["Tiempo (milisegundos)"] = df_resultados["Tiempo (segundos)"] * 1000

df_resultados


,Operación,Tiempo (segundos),Tiempo (milisegundos)
0,Validar 900 folios correctos,0.001842,1.8420
1,Validar 900 folios incorrectos,0.001405,1.4055
2,Crear 300 tickets,0.004456,4.4558
3,Atender 300 tickets,0.193419,193.4187
4,Cerrar 100 tickets,0.041318,41.3177



## 10. Repetir mediciones para obtener un promedio

Una sola medición puede variar por factores externos:

- carga del sistema operativo;
- uso de disco;
- procesos abiertos;
- estado del intérprete de Python;
- primera ejecución de una celda.

Por eso, cuando se quiere comparar con más seriedad, conviene repetir varias veces y calcular un promedio.


In [10]:

def repetir_medicion(descripcion, funcion, repeticiones=5, *args, **kwargs):
    mediciones = []

    for repeticion in range(1, repeticiones + 1):
        medicion = medir_tiempo(descripcion, funcion, *args, **kwargs)

        mediciones.append({
            "repeticion": repeticion,
            "descripcion": descripcion,
            "segundos": medicion["segundos"]
        })

    return pd.DataFrame(mediciones)


df_repeticiones_validacion = repetir_medicion(
    "Validar 900 folios correctos",
    validar_folios_correctos,
    5
)

df_repeticiones_validacion


,repeticion,descripcion,segundos
0,1,Validar 900 folios correctos,0.001538
1,2,Validar 900 folios correctos,0.001369
2,3,Validar 900 folios correctos,0.001481
3,4,Validar 900 folios correctos,0.001792
4,5,Validar 900 folios correctos,0.001414


In [11]:

promedio = df_repeticiones_validacion["segundos"].mean()
minimo = df_repeticiones_validacion["segundos"].min()
maximo = df_repeticiones_validacion["segundos"].max()

print(f"Promedio: {promedio:.8f} segundos")
print(f"Mínimo:   {minimo:.8f} segundos")
print(f"Máximo:   {maximo:.8f} segundos")


Promedio: 0.00151888 segundos
Mínimo:   0.00136940 segundos
Máximo:   0.00179150 segundos



## 11. Comparar cambio de estado directo contra método decorado

Este ejemplo ayuda a explicar una diferencia importante.

No es lo mismo medir:

```python
ticket.estado = "En proceso"
```

que medir:

```python
ticket.atender()
```

La primera operación solo cambia una propiedad validada.  
La segunda operación ejecuta lógica de negocio, decorador, historial y escritura a log.

Por eso, aunque ambas terminan modificando el estado, no cuestan lo mismo.


In [12]:

def cambiar_estado_directo(cantidad=300):
    tickets_locales = crear_tickets(cantidad)

    for ticket in tickets_locales:
        ticket.estado = "En proceso"

    return tickets_locales


def atender_con_metodo(cantidad=300):
    tickets_locales = crear_tickets(cantidad)

    for ticket in tickets_locales:
        ticket.atender()

    return tickets_locales


limpiar_archivo_log()

medicion_directa = medir_tiempo(
    "Cambiar estado directo en 300 tickets",
    cambiar_estado_directo,
    300
)

limpiar_archivo_log()

medicion_metodo = medir_tiempo(
    "Atender 300 tickets con método decorado",
    atender_con_metodo,
    300
)

comparacion_estado = pd.DataFrame([
    {
        "Operación": medicion_directa["descripcion"],
        "Tiempo (segundos)": medicion_directa["segundos"],
        "Tiempo (milisegundos)": medicion_directa["segundos"] * 1000
    },
    {
        "Operación": medicion_metodo["descripcion"],
        "Tiempo (segundos)": medicion_metodo["segundos"],
        "Tiempo (milisegundos)": medicion_metodo["segundos"] * 1000
    }
])

comparacion_estado


,Operación,Tiempo (segundos),Tiempo (milisegundos)
0,Cambiar estado directo en 300 tickets,0.004109,4.109
1,Atender 300 tickets con método decorado,0.201433,201.433



## 12. Interpretación didáctica

Con estos ejercicios se puede explicar que `time.perf_counter()` sirve para medir duración, no para mostrar una hora del día.

En esta aplicación, los resultados ayudan a diferenciar entre:

- operaciones simples, como validar un folio;
- creación de objetos;
- ejecución de métodos de negocio;
- métodos decorados;
- operaciones que además escriben en archivo.

Una conclusión importante para clase:

> Si medimos una función, medimos todo lo que ocurre dentro de ella.  
> Si el método validó, cambió estado, registró historial y escribió en disco, todo eso forma parte del tiempo final.

Por eso, al analizar rendimiento, primero se debe entender qué hace realmente el código.


In [13]:

print("Resumen final")
print("-------------")
print(f"Total de tickets creados durante la sesión: {Ticket.total_tickets_creados()}")

if os.path.exists("registro_acciones.log"):
    print(f"Archivo de log generado: registro_acciones.log")
    print(f"Tamaño actual: {os.path.getsize('registro_acciones.log')} bytes")
else:
    print("No existe archivo de log en este momento.")


Resumen final
-------------
Total de tickets creados durante la sesión: 901
Archivo de log generado: registro_acciones.log
Tamaño actual: 50200 bytes
